# 01 PyTorch 基础：Tensor 创建、常用方法、autograd

这一节重新按“初学者能看懂、以后能回来查”的方式写。

你的目标不是背 API，而是知道：

- 这个方法是干什么的。
- 什么时候该用它。
- 常用参数是什么意思。
- 返回的 Tensor 形状、类型、设备是什么。
- 哪些地方最容易写错。

参考顺序：黑马程序员《神经网络与深度学习》Day01 的 PyTorch 张量创建、全 0/1/指定值张量、线性和随机张量、元素类型转换；细节以 PyTorch 官方文档为准。

## 0. 先说人话：Tensor 到底是什么

Tensor 就是 PyTorch 里的“多维数值容器”。

你可以先这样理解：

| 数学/机器学习里的东西 | PyTorch 里的形状 | 例子 |
|---|---:|---|
| 一个数，标量 | `[]` | loss = 0.35 |
| 一个向量 | `[特征数]` | 一个样本有 4 个特征 |
| 一个矩阵 | `[行, 列]` | 一批表格数据 |
| 一批图片 | `[batch, channel, height, width]` | 32 张 RGB 图片：`[32, 3, 224, 224]` |

PyTorch 里的模型、输入、标签、权重、梯度，基本都用 Tensor 表示。

## 1. 环境检查

推荐运行环境：`D:\\PythonWorkSpace\\anaconda\\envs\\pytorch\\python.exe`。

当前机器上这套环境可以导入 PyTorch。由于你的显卡比较新，当前 PyTorch 的 CUDA 版本可能不能完整支持它，所以本节全部用 CPU 跑，先把基础概念学扎实。

In [1]:
import torch
import numpy as np

print("torch version:", torch.__version__)

# 先固定用 CPU，避免 CUDA 版本和新显卡架构不兼容导致初学阶段被环境问题打断。
device = torch.device("cpu")
print("device:", device)

torch version: 2.10.0
device: cpu


## 2. Tensor 最重要的 5 个属性

以后调试模型，先看这 5 个属性：

| 属性/方法 | 人话解释 | 常见用途 |
|---|---|---|
| `x.shape` / `x.size()` | 张量形状 | 检查维度是否匹配 |
| `x.ndim` / `x.dim()` | 张量有几维 | 判断标量、向量、矩阵还是更高维 |
| `x.dtype` | 元素类型 | 特征通常 `float32`，分类标签常用 `long` |
| `x.device` | 张量在哪个设备上 | CPU/GPU 混用会报错 |
| `x.requires_grad` | 是否需要记录梯度 | 模型参数需要，普通标签不需要 |

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

print("x =\n", x)
print("shape:", x.shape)
print("size():", x.size())
print("ndim:", x.ndim)
print("dtype:", x.dtype)
print("device:", x.device)
print("requires_grad:", x.requires_grad)

## 3. 创建 Tensor：从已有数据创建

### 3.1 `torch.tensor(data, dtype=None, device=None, requires_grad=False)`

作用：把 Python 数字、列表、嵌套列表、NumPy 数组转换成 Tensor。

常用参数：

| 参数 | 说明 |
|---|---|
| `data(原始数据)` | 比如数字、列表、NumPy 数组 |
| `dtype(元素类型)` | 指定元素类型，比如 `torch.float32`、`torch.int64` |
| `device(设备)` | 指定设备，比如 `"cpu"`、`"cuda"` |
| `requires_grad(是否追踪梯度)` | 训练模型参数时才常用 |

初学建议：优先用 `torch.tensor(...)`，清楚、稳定、可读性高。

In [ ]:
a = torch.tensor(10)
b = torch.tensor([1, 2, 3])
c = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
d = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

for name, value in {"a": a, "b": b, "c": c, "d": d}.items():
    print(name, value, "shape=", value.shape, "dtype=", value.dtype, "requires_grad=", value.requires_grad)

### 3.2 `torch.Tensor(...)`、`torch.FloatTensor(...)`、`torch.IntTensor(...)`

这些是早期常见写法，很多课程会讲，所以你要认识它们。

| 写法 | 作用 | 注意 |
|---|---|---|
| `torch.Tensor(data)` | 创建浮点 Tensor | 默认通常是 `float32` |
| `torch.Tensor(2, 3)` | 创建形状为 `[2, 3]` 的未初始化 Tensor | 值是内存里的旧值，不是 0 |
| `torch.FloatTensor(data)` | 创建 `float32` Tensor | 老写法，可读性一般 |
| `torch.IntTensor(data)` | 创建 `int32` Tensor | 分类标签一般更常用 `long/int64` |

初学建议：能看懂这些写法，但自己写新代码时优先用 `torch.tensor(..., dtype=...)`、`torch.zeros(...)`、`torch.ones(...)` 等更明确的函数。

In [ ]:
old_style_1 = torch.Tensor([1, 2, 3])
old_style_2 = torch.FloatTensor([1, 2, 3])
old_style_3 = torch.IntTensor([1, 2, 3])

print(old_style_1, old_style_1.dtype)
print(old_style_2, old_style_2.dtype)
print(old_style_3, old_style_3.dtype)

# 这个只分配空间，不保证里面是什么值。不要把它当全 0 张量。
uninitialized = torch.Tensor(2, 3)
print(uninitialized)

## 4. 创建固定值 Tensor

这类函数用于初始化数据、占位、构造 mask、构造标签等。

| 方法 | 作用 | 常用参数 | 例子 |
|---|---|---|---|
| `torch.zeros(size(形状))` | 创建全 0 Tensor | `dtype(元素类型)`, `device(设备)` | `torch.zeros(2, 3)` |
| `torch.ones(size(形状))` | 创建全 1 Tensor | `dtype(元素类型)`, `device(设备)` | `torch.ones(2, 3)` |
| `torch.full(size(形状), fill_value(填充值))` | 创建指定值 Tensor | `dtype(元素类型)` | `torch.full((2, 3), 7)` |
| `torch.empty(size(形状))` | 只分配空间，不初始化 | `dtype(元素类型)`, `device(设备)` | 很少给初学者直接用 |
| `torch.eye(n(行数), m(列数))` | 创建单位矩阵 | `dtype(元素类型)` | 线性代数里常见 |

注意：`size` 可以写成 `torch.zeros(2, 3)`，也可以写成 `torch.zeros((2, 3))`。

In [ ]:
print("zeros:\n", torch.zeros(2, 3))  # size(形状)
print("ones:\n", torch.ones(2, 3))  # size(形状)
print("full:\n", torch.full((2, 3), 7))  # size(形状), fill_value(填充值)
print("eye:\n", torch.eye(3))  # n(行数)

### 4.1 `xxx_like`：照着别人的形状创建

`zeros_like`、`ones_like`、`full_like` 的意思是：形状、设备、类型默认参考已有 Tensor。

这在写模型时很常用，因为你经常想创建一个“和输入同形状”的 mask、权重或临时变量。

In [ ]:
base = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

print(torch.zeros_like(base))
print(torch.ones_like(base))
print(torch.full_like(base, 9.0))

## 5. 创建线性序列和随机 Tensor

| 方法 | 作用 | 常用参数                                 | 适合场景        |
|---|---|--------------------------------------|-------------|
| `torch.arange(start, end, step)` | 像 Python `range`，左闭右开 | `start(起始值)`, `end(终止值)`, `step(步长)` | 生成整数序列、索引   |
| `torch.linspace(start(起始值), end(终止值), steps(元素个数))` | 在区间内等间隔取点，包含两端 | `start(起始值)`, `end(终止值)`, `steps(分成的元素个数)`     | 画函数、造实验数据   |
| `torch.rand(size(形状))` | `[0, 1)` 均匀分布随机数 | `size(几行几列)`                         | 注意使用随机种子    |
| `torch.randn(size(形状))` | 标准正态分布随机数 N(0, 1) | `size(几行几列)`                               | 深度学习初始化、造噪声 |
| `torch.randint(low(下限), high(上限), size(形状))` | 随机整数，左闭右开 | `low(下限)`, `high(上限)`, `size(几行几列)`                | 随机类别、随机索引   |
| `torch.manual_seed(seed(随机种子值))` | 固定随机种子 | `seed(随机种子值)`                               | 让实验可复现      |

人话区别：`rand` 是 0 到 1 的随机小数；`randn` 是均值约 0、标准差约 1 的随机数，可能为负。

In [2]:
torch.manual_seed(42)  # seed(随机种子值)

print("arange:", torch.arange(0, 10, 2))  # start(起始值), end(终止值), step(步长)
print("linspace:", torch.linspace(0, 1, 5))  # start(起始值), end(终止值), steps(元素个数)
print("rand:\n", torch.rand(2, 3))  # size(形状)
print("randn:\n", torch.randn(2, 3))  # size(形状)
print("randint:", torch.randint(0, 10, (5,)))  # low(下限), high(上限), size(形状)

arange: tensor([0, 2, 4, 6, 8])
linspace: tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
rand:
 tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009]])
randn:
 tensor([[ 1.1561,  0.3965, -2.4661],
        [ 0.3623,  0.3765, -0.1808]])
randint: tensor([7, 6, 9, 6, 3])


## 6. 数据类型 dtype：别小看它

深度学习里最常见的 dtype：

| dtype | 人话解释 | 常见场景 |
|---|---|---|
| `torch.float32` / `torch.float` | 单精度浮点数 | 模型输入、权重、回归标签 |
| `torch.float64` / `torch.double` | 双精度浮点数 | 科学计算多，深度学习较少用 |
| `torch.int64` / `torch.long` | 64 位整数 | 多分类标签、索引 |
| `torch.int32` / `torch.int` | 32 位整数 | 一般整数数据 |
| `torch.bool` | 布尔值 | mask、条件筛选 |

两个非常常见的坑：

- 神经网络输入通常要是浮点数，不是整数。
- `nn.CrossEntropyLoss` 的分类标签通常要是 `torch.long`，不是 one-hot，也不是 float。

### 6.1 类型转换怎么写

| 写法 | 作用 | 人话说明 |
|---|---|---|
| `x.float()` | 转成 `torch.float32` | 神经网络输入最常用 |
| `x.double()` | 转成 `torch.float64` | 精度更高，但训练里不常用 |
| `x.long()` | 转成 `torch.int64` | 多分类标签常用 |
| `x.int()` | 转成 `torch.int32` | 普通整数 |
| `x.bool()` | 转成布尔类型 | mask、条件筛选 |
| `x.to(dtype=torch.float32)` | 用 `to` 指定目标类型 | 写法更统一，也能同时换 device |

注意：这些转换通常会返回一个新的 Tensor，不会原地修改旧 Tensor。也就是说，`x.float()` 只是得到一个 float 版本；如果你想让变量 `x` 以后都变成 float，要写 `x = x.float()`。

In [ ]:
x_int = torch.tensor([1, 2, 3])

x_float = x_int.float()      # int64 -> float32
x_long = x_float.long()      # float32 -> int64
x_double = x_float.double()  # float32 -> float64
x_bool = x_int.bool()        # 非 0 变 True，0 变 False

print("x_int dtype:", x_int.dtype)
print("x_float dtype:", x_float.dtype)
print("x_long dtype:", x_long.dtype)
print("x_double dtype:", x_double.dtype)
print("x_bool dtype:", x_bool.dtype)

# 也可以用 .to(dtype=...)
x_to_float = x_int.to(dtype=torch.float32)
print("x_to_float dtype:", x_to_float.dtype)

# 原来的 x_int 没有被改掉
print("x_int still dtype:", x_int.dtype)

## 7. Tensor 和 NumPy 互转

| 方法 | 作用 | 是否可能共享内存 | 什么时候用 |
|---|---|---:|---|
| `torch.from_numpy(arr)` | NumPy 转 Tensor | 是 | 想保留和 NumPy 的联系 |
| `torch.tensor(arr)` | NumPy/列表 转 Tensor | 否，通常复制数据 | 想要独立 Tensor |
| `torch.as_tensor(arr)` | 尽量不复制地转 Tensor | 可能 | 追求效率但要懂共享风险 |
| `tensor.numpy()` | Tensor 转 NumPy | CPU Tensor 上通常共享 | 画图、传统工具处理 |

人话提醒：如果你不想两个变量互相影响，就用 `.clone()` 或 `torch.tensor(...)` 明确复制。

In [ ]:
arr = np.array([1, 2, 3], dtype=np.float32)

t_shared = torch.from_numpy(arr)
t_copied = torch.tensor(arr)

arr[0] = 100

print("arr:", arr)
print("from_numpy shares memory:", t_shared)
print("torch.tensor copied data:", t_copied)

## 8. 索引和切片：拿出你想要的部分

索引规则和 NumPy 很像。

| 写法 | 说明 |
|---|---|
| `x[0]` | 第 0 行 |
| `x[:, 0]` | 所有行，第 0 列 |
| `x[1:3]` | 第 1 到第 2 行，左闭右开 |
| `x[:2, 1:]` | 前两行，从第 1 列到最后 |
| `x[x > 0]` | 布尔索引，取满足条件的元素 |

注意：Python 从 0 开始计数。`1:3` 包含 1，不包含 3。

In [ ]:
x = torch.arange(12).reshape(3, 4)

print("x =\n", x)
print("x[0] =", x[0])
print("x[:, 0] =", x[:, 0])
print("x[1:3] =\n", x[1:3])
print("x[:2, 1:] =\n", x[:2, 1:])
print("x[x > 5] =", x[x > 5])

## 9. 改变形状：reshape、view、flatten、squeeze、unsqueeze

| 方法 | 作用 | 常用参数 | 人话例子 |
|---|---|---|---|
| `reshape(shape(目标形状))` | 改成指定形状 | `shape(目标形状)` | `[12] -> [3, 4]` |
| `view(shape(目标形状))` | 改成指定形状，但要求内存连续 | `shape(目标形状)` | 老代码常见 |
| `flatten(start_dim(起始维度)=0)` | 拉平成一维或从某一维开始拉平 | `start_dim(起始维度)`, `end_dim(结束维度)` | 图片进全连接层前常用 |
| `unsqueeze(dim(插入位置))` | 在指定位置增加一个维度 | `dim(插入位置)` | `[3] -> [1, 3]` |
| `squeeze(dim(删除位置)=None)` | 删除长度为 1 的维度 | `dim(删除位置)` | `[1, 3, 1] -> [3]` |

核心要求：改形状前后，元素总数必须一样。`12` 个数可以改成 `[3, 4]`，不能改成 `[5, 5]`。

In [ ]:
x = torch.arange(12)

print("original:", x, x.shape)
print("reshape(3, 4):\n", x.reshape(3, 4))  # shape(目标形状)
print("flatten:", x.reshape(3, 4).flatten())  # start_dim(起始维度), end_dim(结束维度)

v = torch.tensor([1, 2, 3])
print("v.shape:", v.shape)
print("v.unsqueeze(0).shape:", v.unsqueeze(0).shape)  # dim(插入位置)
print("v.unsqueeze(1).shape:", v.unsqueeze(1).shape)  # dim(插入位置)

y = torch.zeros(1, 3, 1)
print("y.shape:", y.shape)
print("y.squeeze().shape:", y.squeeze().shape)

### 9.1 `reshape` 和 `view` 的区别

初学阶段先记一句：优先用 `reshape`，它更省心。

- `view` 要求 Tensor 的内存是连续的。
- `reshape` 会尽量返回 view；如果不行，可能复制一份新数据。
- 如果你明确想复制，用 `clone()`。
- 如果你遇到 `view` 报错，可以先试 `reshape` 或 `contiguous().view(...)`。

In [ ]:
x = torch.arange(12).reshape(3, 4)
x_t = x.t()  # 转置后通常不是连续内存

print("x_t.shape:", x_t.shape)
print("is_contiguous:", x_t.is_contiguous())
print("reshape works:", x_t.reshape(12))

# 如果一定要用 view，先 contiguous。
print("contiguous + view works:", x_t.contiguous().view(12))

## 10. 维度交换：transpose、t、permute

| 方法 | 作用 | 适合场景 |
|---|---|---|
| `x.t()` | 只适合二维矩阵转置 | 矩阵行列互换 |
| `x.transpose(dim0(维度0), dim1(维度1))` | 交换两个维度 | 任意维 Tensor 交换两个轴 |
| `x.permute(*dims(维度序列))` | 重新排列所有维度 | 图像通道顺序转换常用 |

例子：图片在 NumPy/Matplotlib 里常见是 `[H, W, C]`，PyTorch CNN 常见是 `[C, H, W]`，这时就会用 `permute`。

In [ ]:
matrix = torch.arange(6).reshape(2, 3)
print("matrix:\n", matrix)
print("matrix.t():\n", matrix.t())

image_hwc = torch.zeros(224, 224, 3)
image_chw = image_hwc.permute(2, 0, 1)
print("HWC:", image_hwc.shape)
print("CHW:", image_chw.shape)

## 11. 广播机制 broadcasting

广播就是：两个形状不同的 Tensor，在规则允许时，PyTorch 自动把小的那个“扩展”成能计算的形状。

最常见例子：一批样本加同一个偏置。

`x.shape = [3, 4]`，`bias.shape = [4]`，相加时 `bias` 会被当成 `[1, 4]`，再广播到 `[3, 4]`。

人话提醒：广播很方便，但也容易让 shape 错误不报错，所以每次写完关键计算都打印一下 shape。

In [ ]:
x = torch.ones(3, 4)
bias = torch.tensor([10, 20, 30, 40])

print("x.shape:", x.shape)
print("bias.shape:", bias.shape)
print(x + bias)

## 12. 常用数学运算

这一节先把常用运算讲清楚。你现在不要急着背所有函数，先记住两个问题：

1. 这个运算是在“每个元素上算”，还是在“某个维度上汇总”？
2. 如果函数里有 `dim`，它到底沿着哪一维算？

| 方法/符号 | 作用 | 常用参数 | 人话解释 |
|---|---|---|---|
| `x + y`、`x - y`、`x * y`、`x / y` | 逐元素运算 | 无，主要看 shape 是否相同或可广播 | 对应位置一个一个算 |
| `x @ y` | 矩阵乘法/向量点积 | 无，主要看矩阵形状是否能相乘 | 线性代数里的矩阵乘法 |
| `torch.matmul(x, y)` | 矩阵乘法/批量矩阵乘法 | `x`, `y` | 和 `@` 类似，函数写法更明确 |
| `x.sum(dim=None, keepdim=False)` | 求和 | `dim`、`keepdim` | 把所有数或某个维度上的数加起来 |
| `x.mean(dim=None, keepdim=False)` | 求平均值 | `dim`、`keepdim` | 把所有数或某个维度上的数求平均 |
| `x.max(dim=None, keepdim=False)` | 求最大值 | `dim`、`keepdim` | 找最大值；指定 `dim` 时还会返回最大值的位置索引 |

注意：`*` 是逐元素乘法，不是矩阵乘法。矩阵乘法用 `@` 或 `torch.matmul`。

In [ ]:
import torch

x = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0]
])

y = torch.tensor([
    [10.0, 20.0],
    [30.0, 40.0]
])

print("x =\n", x)
print("y =\n", y)

print("逐元素乘法 x * y：\n", x * y)
print("矩阵乘法 x @ y：\n", x @ y)
print("函数写法 torch.matmul(x, y)：\n", torch.matmul(x, y))

### 12.1 `dim` 到底是什么意思

`dim` 是 dimension 的缩写，意思是“维度”。

对二维 Tensor 来说：

```python
x.shape = [行数, 列数]
```

- `dim=0`：沿着第 0 维，也就是沿着“行的方向”往下算。结果会把多行压成一行，所以输出长度等于列数。
- `dim=1`：沿着第 1 维，也就是沿着“列的方向”横着算。结果会把多列压成一列，所以输出长度等于行数。

更直白一点：

| 写法 | 在二维表里怎么理解 | 输出形状 |
|---|---|---|
| `x.sum()` | 所有元素全加起来 | 标量 `[]` |
| `x.sum(dim=0)` | 每一列分别求和 | `[列数]` |
| `x.sum(dim=1)` | 每一行分别求和 | `[行数]` |

记忆方法：`dim` 指的是“被压缩掉的那一维”。

In [1]:
import torch

x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

print("x =\n", x)
print("x.shape =", x.shape)

print("所有元素求和 x.sum() =", x.sum())
print("每一列求和 x.sum(dim=0) =", x.sum(dim=0))
print("每一行求和 x.sum(dim=1) =", x.sum(dim=1))

print("每一列平均 x.mean(dim=0) =", x.mean(dim=0))
print("每一行平均 x.mean(dim=1) =", x.mean(dim=1))

x =
 tensor([[1., 2., 3.],
        [4., 5., 6.]])
x.shape = torch.Size([2, 3])
所有元素求和 x.sum() = tensor(21.)
每一列求和 x.sum(dim=0) = tensor([5., 7., 9.])
每一行求和 x.sum(dim=1) = tensor([ 6., 15.])
每一列平均 x.mean(dim=0) = tensor([2.5000, 3.5000, 4.5000])
每一行平均 x.mean(dim=1) = tensor([2., 5.])


### 12.2 `keepdim` 是干什么的

`keepdim` 的意思是：求和、求平均、求最大值之后，要不要保留被压缩掉的维度。

默认是：

```python
keepdim=False
```

也就是压缩完之后，这个维度会消失。

如果设置：

```python
keepdim=True
```

这个维度不会消失，只是长度变成 1。

为什么要保留？主要是为了后面继续做广播运算时形状更容易对齐。

In [ ]:
import torch

x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

row_sum_without_keepdim = x.sum(dim=1)
row_sum_with_keepdim = x.sum(dim=1, keepdim=True)

print("x.shape =", x.shape)
print("x.sum(dim=1).shape =", row_sum_without_keepdim.shape)
print("x.sum(dim=1, keepdim=True).shape =", row_sum_with_keepdim.shape)
print("x.sum(dim=1) =", row_sum_without_keepdim)
print("x.sum(dim=1, keepdim=True) =\n", row_sum_with_keepdim)

### 12.3 `max` 的返回值：最大值和索引

`x.max()` 不指定 `dim` 时，只返回整个 Tensor 中最大的那个数。

`x.max(dim=...)` 指定维度时，会返回两个东西：

| 返回值 | 含义 |
|---|---|
| `values` | 最大值是多少 |
| `indices` | 最大值在该维度上的位置索引 |

这在分类任务里特别常见。模型通常会输出每个类别的分数，我们用 `max(dim=1)` 找到每个样本分数最高的类别。

In [ ]:
import torch

scores = torch.tensor([
    [0.1, 2.5, 0.3],
    [1.2, 0.4, 3.1]
])

print("scores =\n", scores)
print("scores.shape =", scores.shape)
print("整个 Tensor 最大值 scores.max() =", scores.max())

values, indices = scores.max(dim=1)
print("每一行的最大值 values =", values)
print("每一行最大值的位置 indices =", indices)
print("预测类别就是 indices =", indices)

### 12.4 向量点积 dot product 怎么算

向量点积是线性代数里的老朋友：两个同长度向量，对应位置相乘，再把结果加起来。

假设：

$$a = [1, 2, 3]$$

$$b = [4, 5, 6]$$

点积就是：

$$a \cdot b = 1 \times 4 + 2 \times 5 + 3 \times 6 = 32$$

PyTorch 里有三种常见写法：

| 写法 | 说明 | 适合什么时候用 |
|---|---|---|
| `torch.dot(a, b)` | 专门算一维向量点积 | 两个都是 1D Tensor 时最清楚 |
| `(a * b).sum()` | 先逐元素乘法，再求和 | 想看清点积本质时最好理解 |
| `a @ b` | 对一维向量时也是点积 | 和矩阵乘法写法统一 |

注意：`torch.dot` 只适合一维向量。如果是二维矩阵，通常用 `@` 或 `torch.matmul`。

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

dot_1 = torch.dot(a, b)
dot_2 = (a * b).sum()
dot_3 = a @ b

print("torch.dot(a, b):", dot_1)
print("(a * b).sum():", dot_2)
print("a @ b:", dot_3)

print("逐元素乘法 a * b:", a * b)
print("点积结果是一个标量，shape =", dot_1.shape)

### 12.5 点积、逐元素乘法、矩阵乘法别混

| 运算 | 写法 | 输入例子 | 输出 |
|---|---|---|---|
| 逐元素乘法 | `a * b` | `[3]` 和 `[3]` | `[3]` |
| 向量点积 | `torch.dot(a, b)` 或 `a @ b` | `[3]` 和 `[3]` | 标量 `[]` |
| 矩阵乘向量 | `A @ b` | `[2, 3]` 和 `[3]` | `[2]` |
| 矩阵乘矩阵 | `A @ B` | `[2, 3]` 和 `[3, 4]` | `[2, 4]` |

线性层 `nn.Linear` 本质上就和矩阵乘法有关：输入特征和权重做乘法，再加偏置。后面讲 `nn.Module` 时会继续展开。

In [ ]:
A = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])
b = torch.tensor([10.0, 20.0, 30.0])

print("A.shape:", A.shape)
print("b.shape:", b.shape)
print("A @ b:", A @ b)
print("(A @ b).shape:", (A @ b).shape)

## 13. 设备 device：CPU 和 GPU

Tensor 在哪里算，由 `device` 决定。

| 写法 | 作用 |
|---|---|
| `torch.device("cpu")` | 使用 CPU |
| `torch.device("cuda")` | 使用默认 GPU |
| `x.to(device(目标设备))` | 把 Tensor 移到指定设备 |
| `model.to(device(目标设备))` | 把模型参数移到指定设备 |

最常见报错：输入在 CPU，模型在 GPU，或者反过来。原则很简单：模型和数据必须在同一个设备上。

In [ ]:
x = torch.randn(2, 3)
x = x.to(device)

print(x.device)

## 14. autograd 自动求导：先理解这 4 个词

| 词 | 人话解释 |
|---|---|
| `requires_grad=True` | 告诉 PyTorch：这个 Tensor 参与梯度计算 |
| `grad_fn` | 这个 Tensor 是通过什么运算得到的 |
| `loss.backward()` | 从 loss 开始反向传播，自动算梯度 |
| `x.grad` | loss 对 x 的梯度 |

在机器学习里你学过“求导、梯度下降”。PyTorch 的 autograd 就是帮你把链式法则自动做了。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1

print("y:", y)
print("y.grad_fn:", y.grad_fn)

y.backward()
print("x.grad:", x.grad)

手算检查：

$$y = x^2 + 3x + 1$$

$$\frac{dy}{dx} = 2x + 3$$

当 `x = 2` 时，梯度是 `7`，所以上面 `x.grad` 应该等于 `7`。

### 14.1 梯度会累加：为什么要 `zero_grad()`

PyTorch 默认会把每次 `backward()` 得到的梯度累加到 `.grad` 里，而不是自动覆盖。

所以训练模型时，每轮更新前必须先清空旧梯度：

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x ** 2
y1.backward()
print("第一次 backward 后:", x.grad)

y2 = x ** 2
y2.backward()
print("第二次 backward 后，梯度累加了:", x.grad)

x.grad.zero_()
print("手动清零后:", x.grad)

## 15. `detach()`、`no_grad()`、`item()`

| 方法 | 作用 | 常见场景 |
|---|---|---|
| `x.detach()` | 从计算图里分离出来，不再追踪梯度 | 保存预测结果、转 NumPy |
| `with torch.no_grad():` | 代码块内不记录梯度 | 验证、测试、推理 |
| `x.item()` | 把只有一个元素的 Tensor 转成 Python 数字 | 打印 loss |

人话：训练要梯度，评估不要梯度。不要把验证集、测试集也放进计算图里浪费内存。

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * 2

print("y.requires_grad:", y.requires_grad)
print("y.detach().requires_grad:", y.detach().requires_grad)

loss = y.mean()
print("loss as tensor:", loss)
print("loss as Python number:", loss.item())

## 16. 最小线性回归：把前面的概念串起来

这里不堆代码，只看训练循环最小骨架。

任务：生成接近 `y = 3x + 2` 的数据，让模型学出权重和偏置。

你需要看懂这几个对象：

| 对象 | 作用 |
|---|---|
| `nn.Linear(1, 1)` | 线性模型，输入 1 个特征，输出 1 个值 |
| `nn.MSELoss()` | 均方误差，回归任务常用 |
| `torch.optim.SGD(...)` | 随机梯度下降优化器 |
| `model(x)` | 调用模型做前向计算 |
| `loss.backward()` | 自动计算参数梯度 |
| `optimizer.step()` | 根据梯度更新参数 |

In [ ]:
from torch import nn

torch.manual_seed(42)

x = torch.linspace(-3, 3, 200).reshape(-1, 1)  # start(起始值), end(终止值), steps(元素个数)
noise = 0.3 * torch.randn_like(x)
y = 3 * x + 2 + noise

model = nn.Linear(1, 1)  # in_features(输入特征数), out_features(输出特征数)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)  # params(模型参数), lr(学习率)

for epoch in range(100):
    pred = model(x)
    loss = loss_fn(pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("learned weight:", model.weight.item())
print("learned bias:", model.bias.item())
print("final loss:", loss.item())

## 17. 第一节必须掌握的总结

### 17.1 创建 Tensor 时怎么选方法

| 你想做什么 | 推荐方法 |
|---|---|
| 从 Python 列表创建 | `torch.tensor(data(数据))` |
| 指定数据类型 | `torch.tensor(data(数据), dtype=torch.float32(元素类型))` |
| 创建全 0 | `torch.zeros(shape(形状))` |
| 创建全 1 | `torch.ones(shape(形状))` |
| 创建指定值 | `torch.full(shape(形状), value(填充值))` |
| 创建等差整数序列 | `torch.arange(start(起始值), end(终止值), step(步长))` |
| 创建等间隔浮点序列 | `torch.linspace(start(起始值), end(终止值), steps(元素个数))` |
| 创建 0 到 1 随机数 | `torch.rand(shape(形状))` |
| 创建正态分布随机数 | `torch.randn(shape(形状))` |
| 照着已有 Tensor 创建 | `torch.zeros_like(x)` / `torch.ones_like(x)` |

### 17.2 调试 Tensor 时先看什么

按这个顺序看：

1. `shape`：形状对不对。
2. `dtype`：类型对不对。
3. `device`：是不是都在 CPU 或都在 GPU。
4. `requires_grad`：该求导的有没有开，不该求导的有没有关。

### 17.3 训练循环为什么是那几步

1. `pred = model(x)`：前向传播，得到预测。
2. `loss = loss_fn(pred, y)`：计算预测和真实值的差距。
3. `optimizer.zero_grad()`：清空旧梯度。
4. `loss.backward()`：反向传播，计算新梯度。
5. `optimizer.step()`：更新参数。

## 18. 自检题

学完这一节，不要求你默写所有 API，但要能回答：

1. `torch.tensor([1, 2, 3])` 和 `torch.Tensor(2, 3)` 有什么区别？
2. `torch.rand` 和 `torch.randn` 的随机数分布有什么区别？
3. 为什么分类标签经常要转成 `long`？
4. `reshape` 和 `view` 的区别是什么？初学时优先用哪个？
5. `unsqueeze(0)` 和 `unsqueeze(1)` 对一维向量的形状分别有什么影响？
6. 为什么模型和输入 Tensor 必须在同一个 device 上？
7. 为什么每次训练前要 `optimizer.zero_grad()`？
8. `detach()` 和 `torch.no_grad()` 分别适合什么时候用？

## 19. 下一节怎么学

下一节再进入：

- `Dataset`：把样本组织起来。
- `DataLoader`：按 batch 读取样本。
- `nn.Module`：定义自己的神经网络。
- 二分类 MLP：从“会操作 Tensor”过渡到“会训练模型”。